In [1]:
import json
import openai
import requests

client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [2]:
# 함수 정의

def get_popular_movies():
    """인기 영화 목록을 가져온다."""
    return requests.get(f"{BASE_URL}/movies").json()

def get_movie_details(id):
    """영화 상세 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}").json()

def get_movie_credits(id):
    """영화 출연진/제작진 정보를 가져온다."""
    return requests.get(f"{BASE_URL}/movies/{id}/credits").json()

# 함수 매핑
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [3]:
# AI에게 알려줄 도구(tools) 정의
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the cast and crew of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [4]:
# 대화 이력 (메모리)
messages = []
messages.append(
    {
        "role": "system",
        "content": "You are a movie expert agent. Answer questions about movies using the provided tools. Reply in the same language the user uses.",
    }
)

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    message = response.choices[0].message
    process_ai_response(message)

def process_ai_response(message):
    if message.tool_calls:
        # AI의 tool_calls 응답을 messages에 기록
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ],
            }
        )
        # 각 tool_call에 대해 실제 함수 실행
        for tc in message.tool_calls:
            fn_name = tc.function.name
            arguments = tc.function.arguments
            try:
                args = json.loads(arguments)
            except json.JSONDecodeError:
                args = {}
            print(f"Function Calling {fn_name}({args})")

            function_to_run = FUNCTION_MAP.get(fn_name)
            result = function_to_run(**args)

            # 실행 결과를 role: "tool"로 추가
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )
        # 도구 결과를 포함해서 AI 다시 호출
        call_ai()
    else:
        # 일반 텍스트 응답
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")

In [5]:
print("Movie Agent (종료: q)")

while True:
    user_input = input("\nYou: ")
    if user_input in ("q", "quit"):
        break
    else:
        print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

Movie Agent (종료: q)
User: 
AI: Bonjour! Comment puis-je vous aider aujourd'hui avec des questions sur les films?
User: 지금 인기 있는 영화 알려줘
Function Calling get_popular_movies({})
AI: 현재 인기 있는 영화 목록입니다:

1. **[Your Heart Will Be Broken](https://www.themoviedb.org/movie/1523145)**
   - 개봉일: 2026-03-26
   - 장르: 로맨스, 드라마
   - 줄거리: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하고, 주요 괴롭힘꾼인 바르스와 거래를 하게 된다. 그들은 서로를 보호하는 과정에서 진정한 감정을 발전시키지만, 그들의 가족과 동급생들은 이들을 헤어지게 할 이유가 있다.
   - 평점: 6.75
   ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **[Avatar: Fire and Ash](https://www.themoviedb.org/movie/83533)**
   - 개봉일: 2025-12-17
   - 장르: 공상과학, 모험, 판타지
   - 줄거리: 제이크 술리와 네이티리는 새로운 위협에 직면한다. 그들은 서로의 생존을 위해 싸워야 한다.
   - 평점: 7.37
   ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **[Hoppers](https://www.themoviedb.org/movie/1327819)**
   - 개봉일: 2026-03-04
   - 장르: 애니메이션, 가족, 공상과학, 코미디, 모험
   - 줄거리: 동물과 소통할 수 있는 기술을 활용하여 동물 세계의 신비를 발견하는 이야기입니다.
   - 평점: 7.59
   ![포스터](https